# Task 2 — Latent Semantic Analysis (LSA) using SVD

Construct a Words × Documents matrix, apply log-entropy normalization, perform SVD, project words into latent space, and find closest words using cosine similarity.

In [ ]:
import ast
import numpy as np
from collections import Counter

with open("/mnt/data/dataset.txt", "r", encoding="utf-8") as f:
    content = f.read()

tree = ast.parse(content)
dataset = None

for node in tree.body:
    if isinstance(node, ast.Assign):
        for target in node.targets:
            if isinstance(target, ast.Name) and target.id == "dataset":
                dataset = ast.literal_eval(node.value)

if dataset is None:
    raise ValueError("Variable 'dataset' was not found.")

print("Number of documents:", len(dataset))
print("First document:", dataset[0]["filename"])

## Step 1 — Document-Term Matrix Construction

In [ ]:
def makeDocumentTermMatrix(data):
    vocabulary = sorted(
        set(word for doc in data for word in doc["words"])
    )
    word_to_index = {word: i for i, word in enumerate(vocabulary)}

    matrix = np.zeros((len(vocabulary), len(data)), dtype=float)

    for j, doc in enumerate(data):
        counts = Counter(doc["words"])
        for word, count in counts.items():
            matrix[word_to_index[word], j] = count

    return matrix, vocabulary

X, vocabulary = makeDocumentTermMatrix(dataset)

print("Matrix shape:", X.shape)
print("Vocabulary size:", len(vocabulary))
print(vocabulary[:20])

## Log-Entropy Normalization

In [ ]:
def log_entropy_normalization(X):
    # Local factor
    local = np.zeros_like(X, dtype=float)
    mask = X > 0
    local[mask] = 1 + np.log2(X[mask])

    # Row probabilities
    row_sums = X.sum(axis=1, keepdims=True)
    probabilities = np.divide(
        X, row_sums,
        out=np.zeros_like(X, dtype=float),
        where=row_sums != 0
    )

    # Row entropy
    entropy = np.zeros(X.shape[0])
    for i in range(X.shape[0]):
        p = probabilities[i]
        p = p[p > 0]
        if len(p):
            entropy[i] = -np.sum(p * np.log2(p))

    # Global factor
    if X.shape[1] > 1:
        global_factor = 1 - entropy / np.log2(X.shape[1])
    else:
        # One-document corpus: entropy normalization denominator is 0.
        global_factor = np.ones(X.shape[0])

    normalized = local * global_factor[:, None]
    return normalized, local, global_factor, entropy

X_normalized, local_factor, global_factor, entropy = log_entropy_normalization(X)
print("Normalized shape:", X_normalized.shape)

## Step 2 — Dimensionality Reduction via SVD

The assignment asks for k=2. The supplied file currently contains one document, so rank(X) ≤ 1. The notebook detects this rather than inventing a second dimension.

In [ ]:
requested_k = 2

U, S, VT = np.linalg.svd(
    X_normalized,
    full_matrices=False
)

print("U:", U.shape)
print("S:", S.shape)
print("VT:", VT.shape)
print("Singular values:", S)

actual_k = min(requested_k, len(S))

if actual_k < requested_k:
    print(f"WARNING: requested k={requested_k}, but only k={actual_k} is mathematically available.")

U_k = U[:, :actual_k]
S_k = S[:actual_k]
VT_k = VT[:actual_k, :]

print("Usable k:", actual_k)

In [ ]:
# Standard LSA word coordinates
word_vectors = U_k @ np.diag(S_k)
print("Projected word matrix shape:", word_vectors.shape)

## Step 3 — Latent Similarity Mapping

In [ ]:
def cosine_similarity(a, b):
    denominator = np.linalg.norm(a) * np.linalg.norm(b)
    if denominator == 0:
        return 0.0
    return float(np.dot(a, b) / denominator)

def find_closest_words(target_word, top_n=3):
    if target_word not in vocabulary:
        return f"'{target_word}' is not present in the vocabulary."

    target_index = vocabulary.index(target_word)
    target_vector = word_vectors[target_index]

    results = []
    for i, word in enumerate(vocabulary):
        if word == target_word:
            continue
        similarity = cosine_similarity(target_vector, word_vectors[i])
        results.append((word, similarity))

    results.sort(key=lambda x: x[1], reverse=True)
    return results[:top_n]

In [ ]:
for target in ["machine", "data"]:
    print(f"Closest words to '{target}':")
    print(find_closest_words(target, top_n=3))
    print()

## Dataset Limitation

The supplied `dataset.txt` contains one document, so the Words × Documents matrix has one column and cannot have rank 2. Thus genuine two-dimensional LSA cannot be learned from this file alone. A complete multi-document corpus is required for the requested k=2 result and meaningful `machine`/`data` similarity output.